# Volatility forecasts, revisited

**GBM options agent · version 2 · 11 November 2026.** Version 1 is `gbm_options_rl_agent.ipynb` (git tag `v1`), unchanged. History: `VERSIONS.md`.

The options project (`gbm_options_rl_agent.ipynb`, tagged `v1`) compares five forecasts of SPY's next-21-day volatility in one accuracy table. That notebook is left untouched. This one takes the same five forecasts and asks the questions the table cannot answer:

| Section | Question |
|---|---|
| 2 | Is the ranking real, or could the gaps between forecasters be chance? |
| 3 | Is each forecast unbiased, or does it systematically run high or low? |
| 4 | When do they fail: in calm markets or in stressed ones? |
| 5 | Does combining forecasts beat the best single one? |
| 6 | The question that matters to an option seller: can a forecast tell, in advance, when the premium will be paid and when it will be lost? |

Volatility is a far better target for prediction than returns: it clusters and mean-reverts, so the past carries real information about it.

**Data.** A frozen snapshot of SPY, VIX and the 13-week T-bill, 2007 to November 2026. Forecasts are scored from 2010 on, as in the original. The same period was used there, so this is a deeper analysis of the same forecasts, not a fresh test.

In [1]:
# Imports. vol_eval holds the new scoring tools; func_lib supplies the original five forecasters and the chart theme.
import json
import os
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go

for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.isdir(os.path.join(path, 'quantlib')) and path not in sys.path:
        sys.path.append(path)
from quantlib import func_lib
from quantlib import vol_eval as ve

TRADING_DAYS = 252

In [2]:
# Configuration: identical to the original notebook
IV_TICKER, RF_TICKER = '^VIX', '^IRX'
FIRST_FORECAST_DATE = '2010-01-01'
HORIZON = 21                 # trading days
EWMA_LAMBDA = 0.94
LAGS = HORIZON - 1           # neighbouring 21-day targets share 20 days: allow for that in every test

## 1. The five original forecasts, reproduced

In [3]:
df = pd.read_csv(os.path.join('data', 'SPY_vol_snapshot.csv'), index_col=0, parse_dates=True)
df['rf'] = df[RF_TICKER] / 100
df['log_ret'] = np.log(df['Close']).diff()
df = df.dropna(subset=['log_ret', 'rf', IV_TICKER])
log_ret = df['log_ret']

rv = func_lib.daily_variance_proxy(df).fillna(log_ret ** 2)          # range-based daily variance
target = func_lib.forward_realized_variance(log_ret, HORIZON)       # mean squared return over the next 21 days
squared_ret = log_ret ** 2
var_forecasts = {
    'Historical 21d': squared_ret.rolling(21).mean(),
    'EWMA': squared_ret.ewm(alpha=1 - EWMA_LAMBDA, adjust=False).mean(),
    'GARCH(1,1)': func_lib.walk_forward_garch(log_ret, FIRST_FORECAST_DATE, HORIZON),
    'HAR-RV': func_lib.walk_forward_har(rv, target, FIRST_FORECAST_DATE, HORIZON),
    'Implied (VIX)': (df[IV_TICKER] / 100) ** 2 / TRADING_DAYS,
}
oos = df.index >= pd.Timestamp(FIRST_FORECAST_DATE)
features = pd.DataFrame(var_forecasts)[oos]                          # the model inputs: one variance forecast per column
y = target[oos]
original = func_lib.forecast_metrics({k: features[k] for k in features}, y).sort_values('QLIKE')
print(f'{len(df):,} days, {df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}; scored on {int(y.notna().sum()):,} days from {FIRST_FORECAST_DATE}')
print(original.round(3).to_string())

4,993 days, 2007-01-04 to 2026-11-10; scored on 4,217 days from 2010-01-01
                RMSE (vol pts)  MAE (vol pts)  Bias (vol pts)  QLIKE  MZ R²     Obs
Implied (VIX)            8.160          6.041           3.793  0.382  0.215  4217.0
GARCH(1,1)               7.852          5.328           2.177  0.382  0.187  4217.0
HAR-RV                   8.165          5.804           3.004  0.393  0.171  4217.0
EWMA                     8.412          5.053           0.379  0.519  0.129  4217.0
Historical 21d           8.830          5.309           0.004  0.606  0.107  4217.0


## 2. Is the ranking real?

The table above ranks the forecasters by average loss. An average over 4,000 overlapping days hides how uncertain that ranking is. The **Diebold–Mariano test** takes the daily difference in loss between two forecasts and asks whether its mean is distinguishable from zero, allowing for the fact that neighbouring days share most of their 21-day window.

Losses are **QLIKE**: zero for a perfect forecast, and harsher on forecasting too little risk than too much, which is the right asymmetry for anyone selling options.

In [4]:
losses = pd.DataFrame({k: ve.qlike(features[k], y) for k in features}).dropna()
order = losses.mean().sort_values().index.tolist()
best = order[0]
dm = pd.DataFrame(index=order, columns=order, dtype=float)
pv = dm.copy()
for a in order:
    for b in order:
        if a != b:
            r = ve.diebold_mariano(losses[a], losses[b], LAGS)
            dm.loc[a, b], pv.loc[a, b] = r['dm'], r['p']
print('Mean QLIKE:', losses.mean()[order].round(3).to_dict())
print('\nDiebold-Mariano statistic, row vs column (negative: the row forecast is better):')
print(dm.round(2).to_string())
print('\nTwo-sided p-values:')
print(pv.round(3).to_string())
vs_best = pd.DataFrame({'mean QLIKE': losses.mean()[order], f'DM vs {best}': dm[best], 'p': pv[best]})
print(f'\nEach forecast against the best ({best}):')
print(vs_best.round(3).to_string())

Mean QLIKE: {'Implied (VIX)': 0.382, 'GARCH(1,1)': 0.382, 'HAR-RV': 0.393, 'EWMA': 0.519, 'Historical 21d': 0.606}

Diebold-Mariano statistic, row vs column (negative: the row forecast is better):
                Implied (VIX)  GARCH(1,1)  HAR-RV  EWMA  Historical 21d
Implied (VIX)             NaN       -0.04   -0.87 -2.14           -3.54
GARCH(1,1)               0.04         NaN   -1.38 -2.60           -4.04
HAR-RV                   0.87        1.38     NaN -2.19           -3.58
EWMA                     2.14        2.60    2.19   NaN           -2.85
Historical 21d           3.54        4.04    3.58  2.85             NaN

Two-sided p-values:
                Implied (VIX)  GARCH(1,1)  HAR-RV   EWMA  Historical 21d
Implied (VIX)             NaN       0.971   0.385  0.033           0.000
GARCH(1,1)              0.971         NaN   0.168  0.009           0.000
HAR-RV                  0.385       0.168     NaN  0.028           0.000
EWMA                    0.033       0.009   0.028    NaN  

## 3. Is each forecast unbiased?

The **Mincer–Zarnowitz regression** fits `realized = a + b × forecast`. A forecast that is right on average at every level has `a = 0` and `b = 1`. A slope below 1 means the forecast over-reacts: when it is high, what follows is lower than it says. `p_joint` tests both conditions together; a small value rejects unbiasedness.

In [5]:
mz = pd.DataFrame({k: ve.mincer_zarnowitz(features[k], y, LAGS) for k in order}).T
bias = pd.Series({k: float((np.sqrt(features[k] * TRADING_DAYS) - np.sqrt(y * TRADING_DAYS)).mean() * 100) for k in order}, name='mean error (vol pts)')
mz['mean error (vol pts)'] = bias
print(mz[['a', 'b', 'se_b', 'r2', 'p_joint', 'mean error (vol pts)']].round(4).to_string())

                     a       b    se_b      r2  p_joint  mean error (vol pts)
Implied (VIX)   0.0008  0.7399  0.1392  0.2147   0.0128                3.7932
GARCH(1,1)      0.0107  0.5709  0.1141  0.1873   0.0002                2.1770
HAR-RV          0.0104  0.5340  0.1195  0.1709   0.0004                3.0035
EWMA            0.0167  0.4296  0.1182  0.1292   0.0000                0.3786
Historical 21d  0.0197  0.3267  0.1029  0.1067   0.0000                0.0043


## 4. When do they fail?

Each day is placed in a third by where VIX stood among all VIX readings *up to that day* (so the split uses no future information): calm, normal, stressed.

In [6]:
vix_pct = ve.expanding_percentile(df[IV_TICKER])[oos].reindex(losses.index)
state = pd.cut(vix_pct, [0, 1 / 3, 2 / 3, 1.0], labels=['calm', 'normal', 'stressed'])
by_state = losses.groupby(state, observed=True).mean().T[['calm', 'normal', 'stressed']]
print('Mean QLIKE by market state:')
print(by_state.round(3).to_string())
print('\nDays in each state:', state.value_counts().reindex(['calm', 'normal', 'stressed']).to_dict())
err = pd.DataFrame({k: (np.sqrt(features[k] * TRADING_DAYS) - np.sqrt(y * TRADING_DAYS)) * 100 for k in order}).reindex(losses.index)
print('\nMean error in vol points (positive: forecast too high):')
print(err.groupby(state, observed=True).mean().T[['calm', 'normal', 'stressed']].round(2).to_string())

Mean QLIKE by market state:
^VIX             calm  normal  stressed
Historical 21d  0.702   0.578     0.435
EWMA            0.541   0.572     0.386
GARCH(1,1)      0.405   0.409     0.290
HAR-RV          0.418   0.420     0.298
Implied (VIX)   0.375   0.431     0.322

Days in each state: {'calm': 1965, 'normal': 1362, 'stressed': 890}

Mean error in vol points (positive: forecast too high):
^VIX            calm  normal  stressed
Implied (VIX)   2.82    4.04      5.56
GARCH(1,1)      2.81    1.78      1.39
HAR-RV          3.53    2.82      2.12
EWMA           -0.52    0.47      2.22
Historical 21d -0.95    0.08      1.99


## 5. Does combining forecasts help?

Three combinations, all built without looking ahead:

- **Average of all five**, equal weights.
- **VIX with its premium removed**: VIX runs above what follows on average, so scale it by its own past ratio of outcome to forecast.
- **Regression blend** of HAR, GARCH and VIX, with weights refitted each January on past data only.

In [7]:
full = pd.DataFrame(var_forecasts)
combos = {
    'Average of all five': full.mean(axis=1, skipna=False),
    'VIX, premium removed': ve.debias(full['Implied (VIX)'], target, HORIZON),
    'Blend (HAR, GARCH, VIX)': ve.walk_forward_blend({k: full[k] for k in ('HAR-RV', 'GARCH(1,1)', 'Implied (VIX)')}, target,
                                                      FIRST_FORECAST_DATE, HORIZON),
}
for k, v in combos.items():
    features[k] = v[oos]
all_losses = pd.DataFrame({k: ve.qlike(features[k], y) for k in features}).dropna()
rows = {}
for k in all_losses:
    r = ve.diebold_mariano(all_losses[k], all_losses[best], LAGS) if k != best else {'dm': np.nan, 'p': np.nan}
    rows[k] = {'mean QLIKE': all_losses[k].mean(), f'DM vs {best}': r['dm'], 'p': r['p'],
               'mean error (vol pts)': float((np.sqrt(features[k] * TRADING_DAYS) - np.sqrt(y * TRADING_DAYS)).reindex(all_losses.index).mean() * 100)}
combo_table = pd.DataFrame(rows).T.sort_values('mean QLIKE')
print(f'Scored on the {len(all_losses):,} days where every forecast exists')
print(combo_table.round(3).to_string())

Scored on the 3,714 days where every forecast exists
                         mean QLIKE  DM vs Implied (VIX)      p  mean error (vol pts)
VIX, premium removed          0.366               -0.616  0.538                 1.985
Average of all five           0.371               -0.517  0.605                 2.034
Blend (HAR, GARCH, VIX)       0.378               -0.098  0.922                 1.250
Implied (VIX)                 0.382                  NaN    NaN                 3.652
GARCH(1,1)                    0.387                0.279  0.780                 2.148
HAR-RV                        0.396                0.969  0.333                 2.883
EWMA                          0.517                1.938  0.053                 0.408
Historical 21d                0.599                3.226  0.001                 0.014


## 6. Can a forecast tell when the premium will be paid?

An option seller collects **implied volatility minus the volatility that follows**. Averaged over everything it is positive. The useful question is whether it can be predicted day by day.

For each forecaster, the *expected* premium on a day is implied volatility minus that forecast. Days are sorted into thirds by where the expected premium stands among all its values so far. A forecast is useful to a seller if the realized premium is clearly higher, and losses clearly rarer, in its top third than in its bottom third. The pure-VIX forecast cannot be used here (its expected premium is zero by construction), so "VIX level" is shown instead as the naive rule: sell more when VIX is high.

In [8]:
iv = (df[IV_TICKER] / 100)[oos]
realized_vol = np.sqrt(y * TRADING_DAYS)
realized_premium = (iv - realized_vol).dropna()
print(f'Realized premium: mean {100 * realized_premium.mean():.1f} vol points; positive on {100 * (realized_premium > 0).mean():.0f}% of days; '
      f'worst 5% of days average {100 * realized_premium[realized_premium <= realized_premium.quantile(0.05)].mean():.1f} points\n')

signals = {k: iv - np.sqrt(features[k] * TRADING_DAYS) for k in ('HAR-RV', 'GARCH(1,1)', 'EWMA', 'Historical 21d', 'Blend (HAR, GARCH, VIX)')}
signals['VIX level (naive)'] = iv
rows = {}
for k, s in signals.items():
    pct = ve.expanding_percentile(s.dropna()).reindex(realized_premium.index)
    third = pd.cut(pct, [0, 1 / 3, 2 / 3, 1.0], labels=['low', 'middle', 'high'])
    g = realized_premium.groupby(third, observed=True)
    both = pd.concat([s, realized_premium], axis=1).dropna()
    spread = (realized_premium[third == 'high'].mean() - realized_premium[third == 'low'].mean())
    # is the high-minus-low difference real? regress the realized premium on a high-third dummy within high+low days
    sub = realized_premium[third.isin(['high', 'low'])]
    d = (third[sub.index] == 'high').astype(float)
    x = (d - d.mean()).to_numpy()
    slope = float(x @ sub.to_numpy() / (x @ x))
    resid = sub.to_numpy() - sub.mean() - slope * x
    t = slope / np.sqrt(ve.newey_west_variance(x * resid, LAGS) * len(x) / (x @ x) ** 2)
    rows[k] = {'corr with realized': float(both.corr(method='spearman').iloc[0, 1]),
               'premium, low third': 100 * g.mean()['low'], 'premium, high third': 100 * g.mean()['high'],
               'high minus low': 100 * spread, 't': float(t),
               'loss days, low third %': 100 * float((realized_premium[third == 'low'] < 0).mean()),
               'loss days, high third %': 100 * float((realized_premium[third == 'high'] < 0).mean())}
premium_table = pd.DataFrame(rows).T
print('Realized premium (vol points) by third of the EXPECTED premium:')
print(premium_table.round(2).to_string())

Realized premium: mean 3.8 vol points; positive on 85% of days; worst 5% of days average -18.0 points

Realized premium (vol points) by third of the EXPECTED premium:
                         corr with realized  premium, low third  premium, high third  high minus low     t  loss days, low third %  loss days, high third %
HAR-RV                                 0.22                3.24                 4.44            1.20  1.83                   12.97                    16.43
GARCH(1,1)                             0.22                3.23                 4.75            1.51  2.89                   12.24                    14.82
EWMA                                   0.09                3.68                 4.63            0.95  1.46                   15.17                    12.91
Historical 21d                         0.08                3.35                 4.70            1.35  2.20                   15.46                    14.95
Blend (HAR, GARCH, VIX)                0.23          

## 7. Findings

In [9]:
# Written from the numbers above (nothing typed by hand), and saved for the website
second = order[1]
worst = order[-1]
sig_vs_best = [k for k in order[1:] if pv.loc[k, best] < 0.05]
top_combo = combo_table.index[0]
state_ratio = by_state.loc[best, 'stressed'] / by_state.loc[best, 'calm']
best_signal = premium_table['high minus low'].idxmax()
print(f"1. Ranking: {best} has the lowest loss ({losses.mean()[best]:.3f}), then {second} ({losses.mean()[second]:.3f}); {worst} is last ({losses.mean()[worst]:.3f}).")
print(f"   Against {best}, the difference is statistically clear (p < 0.05) for: {', '.join(sig_vs_best) if sig_vs_best else 'none of the others'}.")
print(f"2. Bias: VIX runs {mz.loc['Implied (VIX)', 'mean error (vol pts)']:+.1f} vol points above what follows, with slope {mz.loc['Implied (VIX)', 'b']:.2f}. "
      f"Unbiasedness is rejected at 5% for: {', '.join(k for k in order if mz.loc[k, 'p_joint'] < 0.05) or 'none'}.")
print(f"3. Regimes: {best}'s loss in stressed markets is {state_ratio:.1f}x its loss in calm ones.")
print(f"4. Combining: the lowest loss of all is {top_combo} ({combo_table.loc[top_combo, 'mean QLIKE']:.3f})"
      + (f", against {combo_table.loc[best, 'mean QLIKE']:.3f} for {best} on the same days (DM {combo_table.loc[top_combo, f'DM vs {best}']:.1f}, p {combo_table.loc[top_combo, 'p']:.3f})." if top_combo != best else ': no combination beat the best single forecast.'))
print(f"5. Predicting the premium: the best separator is {best_signal}: {premium_table.loc[best_signal, 'premium, high third']:.1f} vol points in its high third against "
      f"{premium_table.loc[best_signal, 'premium, low third']:.1f} in its low third (t {premium_table.loc[best_signal, 't']:.1f}); "
      f"losing days {premium_table.loc[best_signal, 'loss days, high third %']:.0f}% against {premium_table.loc[best_signal, 'loss days, low third %']:.0f}%.")
h = premium_table.loc['HAR-RV']
print(f"   The original project's own signal (implied minus the HAR forecast): {h['premium, high third']:.1f} against {h['premium, low third']:.1f} (t {h['t']:.1f}); "
      f"losing days {h['loss days, high third %']:.0f}% in its high third against {h['loss days, low third %']:.0f}% in its low third.")

J = lambda t: json.loads(t.astype(float).to_json(orient='index'))
summary = {'first_date': FIRST_FORECAST_DATE, 'last_date': str(y.dropna().index[-1].date()), 'days': int(len(losses)), 'horizon': HORIZON,
           'order': order, 'best': best, 'mean_qlike': losses.mean()[order].to_dict(),
           'dm_vs_best': {k: {'dm': float(dm.loc[k, best]), 'p': float(pv.loc[k, best])} for k in order[1:]},
           'mz': J(mz[['a', 'b', 'se_b', 'r2', 'p_joint', 'mean error (vol pts)']]),
           'by_state': J(by_state[['calm', 'normal', 'stressed']]), 'combos': J(combo_table.rename(columns={f'DM vs {best}': 'dm'})),
           'premium': J(premium_table), 'premium_overall': {'mean': float(100 * realized_premium.mean()), 'positive_share': float((realized_premium > 0).mean())}}
with open(os.path.join('data', 'volatility_revisited_summary.json'), 'w') as f:
    json.dump(summary, f, indent=1)

1. Ranking: Implied (VIX) has the lowest loss (0.382), then GARCH(1,1) (0.382); Historical 21d is last (0.606).
   Against Implied (VIX), the difference is statistically clear (p < 0.05) for: EWMA, Historical 21d.
2. Bias: VIX runs +3.8 vol points above what follows, with slope 0.74. Unbiasedness is rejected at 5% for: Implied (VIX), GARCH(1,1), HAR-RV, EWMA, Historical 21d.
3. Regimes: Implied (VIX)'s loss in stressed markets is 0.9x its loss in calm ones.
4. Combining: the lowest loss of all is VIX, premium removed (0.366), against 0.382 for Implied (VIX) on the same days (DM -0.6, p 0.538).
5. Predicting the premium: the best separator is VIX level (naive): 4.9 vol points in its high third against 2.7 in its low third (t 2.5); losing days 18% against 16%.
   The original project's own signal (implied minus the HAR forecast): 4.4 against 3.2 (t 1.8); losing days 16% in its high third against 13% in its low third.


## 8. What this does not show

- **One market.** SPY only; single stocks and other asset classes behave differently.
- **The same period as the original.** These forecasts were already scored on 2010 onward, so nothing here is a fresh out-of-sample result.
- **VIX is not a 21-trading-day at-the-money implied volatility.** It is a 30-calendar-day figure built from the whole strip of options, which is why it carries skew as well as the premium.
- **Realized volatility is measured from daily closes.** Intraday data would give a sharper target and could change close rankings.
- **Predicting the premium is not the same as earning it.** Section 6 measures volatility points, before option spreads, margin and the path a short option takes on the way to expiry.